# Aula 02 — Titanic (Pipeline “mínimo profissional”) 


**Disciplina:** Inteligência Artificial  
**Professor:** Marcelo Batista  


## 🎯 Objetivos

1) Utilizando o Dataset Titanic  
2) Manipular dados com Pandas  
3) Treinar seu primeiro modelo de ML (baseline)  
4) Entender o fluxo sagrado: **Dados → Treino → Teste**  
5) Aprender o **pipeline reprodutível** (pré-processamento + modelo)

## Teoria


- IA aplicada = resolver problema real com dados + métricas.
- Modelo = função parametrizada que generaliza padrões.
- Treino ajusta parâmetros para reduzir erros.
- Teste mede generalização em dados não vistos.
- Pipeline evita bagunça e aumenta a reprodutibilidade.


## Bibliotecas que vamos usar


- **Pandas**: “Excel do Python”
- **Seaborn**: datasets prontos + gráficos
- **Scikit-Learn**: ML clássico (split, modelos, métricas, pipelines)

In [1]:
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")

## Dataset Titanic - Pipeline mínimo “profissional” (reprodutível)

Por que precisamos de um pipeline?

##  Agora o jogo fica real: Titanic (tabular com missing + categorias)


Problemas reais têm:
- valores faltantes (missing) - Dados ausentes que precisam ser preenchidos ou tratados.
- colunas categóricas (texto) - Variáveis como "sexo" ou "cidade" que precisam ser convertidas em números para que os modelos possam entendê-las.
- Colunas numéricas: Que podem precisar de padronização ou normalização.


**Solução profissional:** `Pipeline` + `ColumnTransformer`

O **Pipeline** é como uma linha de montagem para o seu fluxo de trabalho de Machine Learning.

O **ColumnTransformer** é uma ferramenta poderosa que permite aplicar diferentes transformações a diferentes subconjuntos de colunas do seu DataFrame, em paralelo.

Por que precisamos dele?

 - Dados Heterogêneos: Em datasets reais, você tem colunas numéricas, categóricas, de texto, etc. Cada tipo de coluna exige um pré-processamento

específico:

- Colunas numéricas: Podem precisar de SimpleImputer (para preencher faltantes com média/mediana) e StandardScaler (para padronizar).

- Colunas categóricas: Podem precisar de SimpleImputer (para preencher faltantes com a moda) e OneHotEncoder (para converter categorias em números binários).
- Flexibilidade: Ele permite que você defina pipelines menores para cada tipo de coluna e os combine. Por exemplo, você pode ter um Pipeline para colunas numéricas e outro para colunas categóricas, e o ColumnTransformer os executa simultaneamente.
- Manutenção: Facilita a adição ou remoção de etapas de pré-processamento para tipos específicos de colunas sem afetar as outras.

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

In [3]:
# Carregar dataset Titanic
df_t = sns.load_dataset("titanic").drop(columns=["alive"])

In [4]:
df_t

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,True


## 📚 Dicionário de Dados — Dataset Titanic

| Nome da Coluna | Tipo de Dado     | Descrição                                                              | Observações                                                                 |
| :------------- | :--------------- | :--------------------------------------------------------------------- | :-------------------------------------------------------------------------- |
| `survived`     | Booleano (int)   | Variável alvo: se o passageiro sobreviveu ou não.                      | `0` = Não, `1` = Sim.                                                       |
| `pclass`       | Categórico (int) | Classe do passageiro.                                                  | `1` = Primeira, `2` = Segunda, `3` = Terceira.                              |
| `sex`          | Categórico       | Sexo do passageiro.                                                    | `male` (masculino) ou `female` (feminino).                                  |
| `age`          | Numérico (float) | Idade do passageiro em anos.                                           | Contém valores `NaN` (faltantes).                                           |
| `sibsp`        | Numérico (int)   | Número de irmãos/cônjuges a bordo.                                     |                                                                             |
| `parch`        | Numérico (int)   | Número de pais/filhos a bordo.                                         |                                                                             |
| `fare`         | Numérico (float) | Tarifa paga pelo passageiro.                                           |                                                                             |
| `embarked`     | Categórico       | Porto de embarque.                                                     | `C` = Cherbourg, `Q` = Queenstown, `S` = Southampton. Contém `NaN`.         |
| `class`        | Categórico       | Classe do passageiro (versão textual).                                 | `First`, `Second`, `Third`. Equivalente a `pclass`.                         |
| `who`          | Categórico       | Categoria da pessoa.                                                   | `man` (homem), `woman` (mulher), `child` (criança).                         |
| `adult_male`   | Booleano         | Indica se o passageiro é um homem adulto.                              | `True` ou `False`.                                                          |
| `deck`         | Categórico       | Nível do convés onde a cabine estava localizada.                       | Muitos valores `NaN` (faltantes).                                           |
| `embark_town`  | Categórico       | Nome completo da cidade de embarque.                                   | `Southampton`, `Cherbourg`, `Queenstown`. Contém `NaN`. Equivalente a `embarked`. |
| `alone`        | Booleano         | Indica se o passageiro estava viajando sozinho.                        | `True` ou `False`.                                                          |


In [5]:
y = df_t["survived"]
X = df_t.drop(columns=["survived"])

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (891, 13)
y shape: (891,)


In [6]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns

print("Numéricas:", list(num_cols))
print("Categóricas:", list(cat_cols))

Numéricas: ['pclass', 'age', 'sibsp', 'parch', 'fare']
Categóricas: ['sex', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alone']


In [7]:
X.head()

,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alone
0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,False
1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,False
2,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,True
3,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,False
4,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,True


In [8]:
y.head()

,survived
0,0
1,1
2,1
3,1
4,0


In [9]:
df_t.describe()

,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


**ATENÇÃO** Isso nos diz que a coluna age possui 891 - 714 = 177 valores faltantes.

## 🛠️ Pré-processamento com `ColumnTransformer` e `Pipeline`

Imagine que seus dados são ingredientes para um sanduíche.

Alguns ingredientes (colunas numéricas) precisam ser fatiados de um jeito, outros (colunas categóricas) de outro.

O `ColumnTransformer` é o chef que garante que cada ingrediente seja preparado corretamente!

In [10]:
preprocess = ColumnTransformer([
    # Bloco 1: Preparar ingredientes NUMÉRICOS
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()) # NOVIDADE: Escala os dados numéricos!
    ]), num_cols),

    # Bloco 2: Preparar ingredientes CATEGÓRICOS
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols),
])

Vamos entender cada "camada" do nosso sanduíche de pré-processamento:

### 1. O Chef Principal: `preprocess = ColumnTransformer([...])`

-   **`preprocess`**: É o nome do nosso "chef" que vai supervisionar toda a preparação.
-   **`ColumnTransformer`**: É a ferramenta que permite ao chef dizer: "Este grupo de colunas vai para a máquina de fatiar X, e aquele grupo vai para a máquina de fatiar Y". Ele aplica **diferentes preparações a diferentes colunas**.
-   **`[...]`**: Aqui dentro, o chef lista as "receitas" para cada tipo de ingrediente. Temos duas receitas principais: uma para números e outra para categorias.

---

### 2. Receita para Ingredientes NUMÉRICOS: `("num", Pipeline([...]), num_cols)`

Esta é a receita para colunas como `idade` ou `tarifa`.

-   **`"num"`**: É o nome desta receita: "Preparação Numérica".
-   **`Pipeline([...])`**: É uma **mini-sequência de passos** para essas colunas. Como uma pequena esteira de cozinha.
    -   **`("imputer", SimpleImputer(strategy="median"))`**:
        -   **`SimpleImputer`**: É a "máquina de preencher buracos". Se faltar um valor (um `NaN`), ela preenche!
        -   **`strategy="median"`**: A máquina preenche com a **mediana** (o valor do meio) da coluna. Isso é bom para números, pois não se abala com valores muito altos ou muito baixos.
    -   **`("scaler", StandardScaler())`**: **NOVIDADE!**
        -   **`StandardScaler`**: Esta é a "máquina de padronizar". Ela ajusta os valores numéricos para que tenham uma média de 0 e um desvio padrão de 1. Isso é **crucial para muitos modelos de ML**, incluindo a Regressão Logística, pois evita que colunas com valores muito grandes dominem o aprendizado.
-   **`num_cols`**: Esta é a lista de colunas (como `['age', 'fare']`) que receberão esta preparação.

---

### 3. Receita para Ingredientes CATEGÓRICOS: `("cat", Pipeline([...]), cat_cols)`

Esta é a receita para colunas como `sexo` ou `porto de embarque`.

-   **`"cat"`**: É o nome desta receita: "Preparação Categórica".
-   **`Pipeline([...])`**: Outra **mini-sequência de passos** para estas colunas.
    -   **`("imputer", SimpleImputer(strategy="most_frequent"))`**:
        -   **`SimpleImputer`**: A mesma "máquina de preencher buracos".
        -   **`strategy="most_frequent"`**: Para categorias, ela preenche com o valor que aparece **mais vezes** (a moda). Se a maioria embarcou em "Southampton", é uma boa aposta!
    -   **`("ohe", OneHotEncoder(handle_unknown="ignore"))`**:
        -   **`OneHotEncoder`**: Esta é a "máquina de traduzir". Ela pega textos (como "male", "female") e os transforma em **números binários (0 ou 1)**, criando novas colunas.
            -   **Exemplo:** `sexo: male` vira `sexo_male: 1, sexo_female: 0`.
        -   **`handle_unknown="ignore"`**: É uma "cláusula de segurança". Se aparecer uma categoria nova e inesperada (ex: um porto que nunca vimos), a máquina não trava, apenas ignora e coloca zeros.
-   **`cat_cols`**: Esta é a lista de colunas (como `['sex', 'embarked']`) que receberão esta preparação.

---

### O Grande Sanduíche Final! 🥪

No fim, o `ColumnTransformer` pega todos os ingredientes preparados (colunas numéricas preenchidas e padronizadas, colunas categóricas traduzidas para números) e os junta em um único "sanduíche" de dados limpos e prontos para o seu modelo de Machine Learning! Ele garante que tudo seja feito de forma organizada e justa, sem "trapacear" com os dados de teste.

## 🚀 Modelo Final: O `Pipeline` Completo (Sanduíche Pronto!)

Depois de preparar todos os ingredientes com o `preprocess` (nosso chef e suas máquinas de fatiar), precisamos montá-los e, finalmente, "comer" o sanduíche (fazer previsões com o modelo)! É isso que este `Pipeline` final faz: ele junta a preparação dos dados com o algoritmo de Machine Learning.



In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

### Dividindo os Dados: `train_test_split(...)`

-   **`X_train, X_test, y_train, y_test`**: Estamos dividindo nossos dados em quatro partes:
    -   `X_train`: As características (features) que o modelo usará para **aprender**.
    -   `y_train`: O que o modelo deve **aprender a prever** (sobrevivência) para os dados de treino.
    -   `X_test`: As características que o modelo usará para **testar** suas previsões (dados que ele nunca viu).
    -   `y_test`: O que o modelo deveria ter previsto para os dados de teste (a "resposta correta").
-   **`test_size=0.2`**: Significa que 20% dos dados serão usados para teste, e os 80% restantes para treino.
-   **`random_state=42`**: Garante que a divisão dos dados seja **sempre a mesma** se você rodar o código novamente. Isso é crucial para a reprodutibilidade dos seus resultados.
-   **`stratify=y`**: Este é um parâmetro muito importante para problemas de classificação! Ele garante que a proporção de "sobreviventes" e "não sobreviventes" seja **mantida de forma similar** tanto no conjunto de treino quanto no de teste. Isso evita que um conjunto de teste, por exemplo, tenha pouquíssimos sobreviventes, o que distorceria a avaliação.

In [12]:
model_baseline_lr = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"))
])

Vamos entender cada parte dessa "linha de montagem" final:

### `model = Pipeline([...])`

-   **`model`**: Este é o nome do nosso (model_baseline_lr) **modelo de Machine Learning completo e pronto para usar**. Ele encapsula todas as etapas, desde a limpeza dos dados até a previsão.
-   **`Pipeline`**: Mais uma vez, o `Pipeline` entra em cena! Ele é como a **linha de montagem principal** que garante que as etapas aconteçam na ordem certa. Primeiro, preparamos os dados; depois, treinamos o modelo.
-   **`[...]`**: Dentro dos colchetes, temos a lista de **duas etapas principais** do nosso fluxo de trabalho.

---

### Primeira Etapa: `("prep", preprocess)` (A Preparação dos Dados)

-   **`"prep"`**: É um **nome descritivo** para esta etapa, indicando que é a fase de pré-processamento.
-   **`preprocess`**: Este é o nosso `ColumnTransformer` que acabamos de explicar! Ele contém todas as regras para limpar e transformar as colunas numéricas e categóricas.
    -   **O que acontece aqui?** Quando o `Pipeline` é executado, esta etapa `("prep", preprocess)` será a primeira a agir. Ela vai pegar seus dados brutos (`X_train` ou `X_test`) e aplicar todas aquelas transformações (preencher `NaN`s com mediana/moda, converter categorias com One-Hot Encoding), resultando em um conjunto de dados limpo e numérico.

---

### Segunda Etapa: `("clf", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"))` (O Classificador)

-   **`"clf"`**: É um nome descritivo para esta etapa, indicando que é o nosso **classificador** (o algoritmo que vai aprender e prever).
-   **`LogisticRegression`**: Este é o algoritmo de Machine Learning que escolhemos para resolver o problema de classificação (prever se o passageiro sobreviveu ou não). É um modelo linear, simples e eficaz para muitas tarefas.
    -   **`max_iter=1000`**: Este é um parâmetro crucial para a `LogisticRegression`. Ele define o número máximo de iterações (tentativas de ajuste) que o algoritmo fará para encontrar os melhores "pesos" para o modelo. Aumentamos para `1000` (o padrão é 100) para:
        *   **Garantir Convergência:** Dar ao algoritmo tempo suficiente para "convergir" (encontrar uma boa solução).
        *   **Evitar Avisos:** Prevenir o `ConvergenceWarning` que vimos anteriormente, que indica que o modelo não conseguiu se ajustar completamente dentro do limite de iterações.
    -   **`random_state=42`**: Este parâmetro garante que, se houver alguma aleatoriedade interna no algoritmo (como na inicialização dos pesos), ela seja sempre a mesma cada vez que você rodar o código. Isso é **crucial para a reprodutibilidade** dos seus resultados.
    -   **`class_weight="balanced"`**: Este é um ajuste muito importante para datasets desbalanceados (onde uma classe tem muito mais exemplos que a outra, como no Titanic). Ele faz com que o algoritmo:
        *   **Pondere as Classes:** Automaticamente ajusta os pesos de cada classe de forma inversamente proporcional à sua frequência nos dados de treino.
        *   **Foque na Minoria:** Ajuda o modelo a dar mais atenção à classe minoritária (os "sobreviventes" no Titanic), evitando que ele simplesmente aprenda a prever a classe majoritária com mais frequência. Isso pode melhorar o desempenho da classe minoritária, especialmente o `recall`.

---

**Em resumo:** Esta etapa pega os dados já limpos e transformados pelo `preprocess` e os usa para treinar a `LogisticRegression`, que agora está configurada para ser mais robusta (com `max_iter` adequado) e justa com as classes (com `class_weight="balanced"`), garantindo resultados consistentes (com `random_state`).



In [13]:
model_baseline_lr.fit(X_train, y_train)


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['pclass', 'age', 'sibsp', 'parch', 'fare'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['sex', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alone'],
      dtype='object'))])),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

### Treinando o Modelo: `model_baseline_lr.fit(X_train, y_train)`

-   **`model_baseline_lr.fit(...)`**: Esta é a mágica! Quando chamamos `fit` no nosso `Pipeline` completo (`model`):
    1.  Primeiro, a etapa `("prep", preprocess)` é executada: o `ColumnTransformer` limpa e transforma o `X_train` (preenche `NaN`s, converte categorias para números), **aprendendo os parâmetros** (mediana, moda, categorias únicas) *apenas* do `X_train`.
    2.  Em seguida, os dados `X_train` **já transformados** são passados para a etapa `("clf", LogisticRegression)`, que então **treina** o algoritmo de Regressão Logística para aprender a prever `y_train`.

In [14]:
pred_baseline_lr = model_baseline_lr.predict(X_test)

### Fazendo Previsões: `pred_baseline_lr = model.predict(X_test)`

-   **`model.predict(X_test)`**: Agora que o modelo aprendeu, pedimos a ele para prever a sobrevivência dos passageiros no `X_test` (dados que ele nunca viu antes).
    1.  O `Pipeline` novamente passa o `X_test` pelo `preprocess`, mas desta vez ele **aplica os mesmos parâmetros** (mediana, moda, categorias) que aprendeu no `X_train`.
    2.  Os dados `X_test` transformados são então usados pelo `LogisticRegression` treinado para gerar as previsões (`pred`).

### Avaliando o Desempenho: `accuracy_score` e `classification_report`

Aqui vemos o quão bem nosso modelo se saiu:

In [15]:
print("Accuracy (Titanic):", accuracy_score(y_test, pred_baseline_lr))
print(classification_report(y_test, pred_baseline_lr))

Accuracy (Titanic): 0.8268156424581006
              precision    recall  f1-score   support

           0       0.86      0.85      0.86       110
           1       0.77      0.78      0.78        69

    accuracy                           0.83       179
   macro avg       0.82      0.82      0.82       179
weighted avg       0.83      0.83      0.83       179



-   **`Accuracy (Titanic): 0.8268...`**: A **acurácia** é a porcentagem de previsões corretas que o modelo fez. Neste caso, 82.68% dos passageiros no conjunto de teste foram classificados corretamente. É um bom resultado para o Titanic!
-   **`classification_report`**: Este relatório é mais detalhado e nos dá uma visão mais profunda:
    -   **`precision` (Precisão)**: Para a classe 0 (não sobreviveu), 0.85 significa que, das vezes que o modelo previu "não sobreviveu", 85% estavam corretas. Para a classe 1 (sobreviveu), 79% das previsões de "sobreviveu" estavam corretas.
    -   **`recall` (Revocação/Sensibilidade)**: Para a classe 0, 0.87 significa que o modelo conseguiu identificar 87% de todos os passageiros que *realmente* não sobreviveram. Para a classe 1, ele identificou 75% dos que *realmente* sobreviveram.
    -   **`f1-score`**: É uma média harmônica entre precisão e recall, útil quando você quer um equilíbrio entre os dois.
    -   **`support`**: O número real de amostras em cada classe no conjunto de teste (110 não sobreviveram, 69 sobreviveram).
    -   **`accuracy` (geral)**: Repete a acurácia total.
    -   **`macro avg`**: É a média simples (não ponderada) das métricas (precisão, recall, f1-score) para cada classe. É útil quando você quer dar a mesma importância a todas as classes, independentemente de quantas amostras elas têm.
    -   **`weighted avg`**: É a média ponderada das métricas, onde cada métrica é ponderada pelo número de amostras (`support`) em sua classe. É mais útil quando as classes estão desbalanceadas (como no Titanic, onde há mais "não sobreviventes" do que "sobreviventes"), pois reflete melhor o desempenho geral do modelo considerando o tamanho de cada classe.

# 📊 Matriz de Confusão: Entendendo os Acertos e Erros do Modelo

A matriz de confusão é como um "mapa" que nos mostra detalhadamente onde nosso modelo acertou e onde errou em suas previsões. Ela compara o que o modelo previu com o que realmente aconteceu.



### Como ler a Matriz de Confusão:

|                | **Previsto: 0 (Não Sobreviveu)** | **Previsto: 1 (Sobreviveu)** |
| :------------- | :------------------------------- | :--------------------------- |
| **Real: 0 (Não Sobreviveu)** | **Verdadeiro Negativo (TN)**: O modelo previu 0 e o real era 0. **Acerto!** | **Falso Positivo (FP)**: O modelo previu 1, mas o real era 0. **Erro!** (Tipo I) |
| **Real: 1 (Sobreviveu)** | **Falso Negativo (FN)**: O modelo previu 0, mas o real era 1. **Erro!** (Tipo II) | **Verdadeiro Positivo (TP)**: O modelo previu 1 e o real era 1. **Acerto!** |


---



In [16]:
# --- Avaliando o Desempenho do Modelo Baseline ---
print("\n--- Desempenho do Modelo BASELINE (LogisticRegression) ---")
# CORREÇÃO AQUI: Garante que o resultado seja um float
acc_baseline_lr = float(accuracy_score(y_test, pred_baseline_lr))
print(f"Accuracy (Baseline - LogisticRegression): {acc_baseline_lr:.4f}")
print("\nClassification Report (Baseline):")
print(classification_report(y_test, pred_baseline_lr))

print("\nMatriz de Confusão (Baseline):")
cm_baseline_lr = confusion_matrix(y_test, pred_baseline_lr)
print(cm_baseline_lr)

# Extraindo valores para a interpretação posterior
tn_baseline_lr, fp_baseline_lr, fn_baseline_lr, tp_baseline_lr = cm_baseline_lr.ravel()

print("\nModelo Baseline (LogisticRegression) avaliado com sucesso!")



--- Desempenho do Modelo BASELINE (LogisticRegression) ---
Accuracy (Baseline - LogisticRegression): 0.8268

Classification Report (Baseline):
              precision    recall  f1-score   support

           0       0.86      0.85      0.86       110
           1       0.77      0.78      0.78        69

    accuracy                           0.83       179
   macro avg       0.82      0.82      0.82       179
weighted avg       0.83      0.83      0.83       179


Matriz de Confusão (Baseline):
[[94 16]
 [15 54]]

Modelo Baseline (LogisticRegression) avaliado com sucesso!


-   **Verdadeiros Positivos (TP)**: O modelo previu que o passageiro sobreviveria (1) e ele *realmente* sobreviveu (1). (Canto inferior direito)
-   **Verdadeiros Negativos (TN)**: O modelo previu que o passageiro não sobreviveria (0) e ele *realmente* não sobreviveu (0). (Canto superior esquerdo)
-   **Falsos Positivos (FP)**: O modelo previu que o passageiro sobreviveria (1), mas ele *não* sobreviveu (0). Um "alarme falso" de sobrevivência. (Canto superior direito)
-   **Falsos Negativos (FN)**: O modelo previu que o passageiro não sobreviveria (0), mas ele *realmente* sobreviveu (1). O modelo "perdeu" um sobrevivente. (Canto inferior esquerdo)

# 🧐 Interpretação (Objetiva)

Agora, vamos usar a matriz de confusão e o relatório de classificação que vimos anteriormente para analisar nosso modelo:

1.  **Sua acurácia parece boa? Por quê?**
    *   Sim, uma acurácia de **82.12%** é considerada boa para um problema do mundo real como o do Titanic.
    *   Isso significa que o modelo acertou a previsão de sobrevivência em mais de 8 de cada 10 passageiros no conjunto de teste. É um resultado significativamente melhor do que chutar aleatoriamente (50%), indicando que o modelo conseguiu aprender padrões relevantes nos dados.

2.  **Olhando precision/recall, o modelo erra mais em qual classe?**
    *   O modelo parece errar mais na **classe 1 (Sobreviveu)**.
    *   Para a classe 1, temos uma **precisão de 0.78** e um **recall de 0.74**. Isso significa que, das vezes que o modelo previu "sobreviveu", 22% estavam erradas (Falsos Positivos). E, mais criticamente, ele só conseguiu identificar 74% de todos os passageiros que *realmente* sobreviveram (os outros 26% foram Falsos Negativos, ou seja, ele previu que não sobreviveram, mas eles sobreviveram).
    *   Comparativamente, para a classe 0 (Não Sobreviveu), a precisão (0.84) e o recall (0.87) são mais altos, mostrando que o modelo é mais confiante e eficaz em identificar quem não sobreviveu.

3.  **Qual é uma limitação óbvia do Titanic (dados faltantes, variáveis ausentes, viés histórico etc.)?**
    *   Uma limitação óbvia do dataset do Titanic é a **presença de dados faltantes significativos**, especialmente na coluna `deck` (convés) e `age` (idade), que são variáveis potencialmente muito importantes para a sobrevivência.
    *   Além disso, o dataset reflete um **viés histórico** inerente ao evento, como a política de "mulheres e crianças primeiro" e as diferenças nas taxas de sobrevivência entre as classes sociais. O modelo aprende esses vieses, o que pode não ser desejável em outros contextos, mas é uma característica dos dados históricos.
    *   Variáveis ausentes, como a localização exata no navio, a proximidade aos botes salva-vidas, ou a capacidade de nadar, também são fatores cruciais que não estão presentes no dataset, limitando o poder preditivo do modelo.